# Lab 06: DCGAN on MNIST

**Module 06** | Read `notes/06-dcgan-gan-training.pdf` before running this notebook.

Apply DCGAN architectural rules (strided convolutions, batch norm, no pooling) and compare sample quality to the vanilla GAN lab.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## DCGAN on MNIST

Deep Convolutional GANs (DCGANs) replace fully connected layers with strided convolutions in the discriminator and transposed convolutions in the generator. Radford et al. distilled architectural rules that stabilize GAN training: batch normalization in both networks (except the generator output and discriminator input), LeakyReLU activations, strided convolutions instead of pooling, and no fully connected hidden layers.

Compared with the shallow MLP GAN in Lab 05, DCGANs exploit spatial structure in 28x28 digits and typically produce sharper samples.


### Step 1: Load and inspect MNIST

MNIST images are scaled to `[-1, 1]` so they match the generator's `Tanh` output range. We use batch size 128 and train for six epochs, enough to see digit-like structure without long GPU time.

The cell below downloads MNIST (if needed), builds a DataLoader, and prints dataset size plus statistics from one real batch.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "datasets" / "mnist"
BATCH_SIZE = 128
LATENT_DIM = 100
EPOCHS = 6
LR = 2e-4

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print("Dataset summary")
print(f"  Train images: {len(train_set):,}")
print(f"  Image shape:  {tuple(train_set[0][0].shape)}")
print(f"  Batches/epoch: {len(loader)}")
print(f"  Latent dim:   {LATENT_DIM}")

real_batch, real_labels = next(iter(loader))
print("\nOne real batch (before .to(device))")
print(f"  Batch shape:  {tuple(real_batch.shape)}")
print(f"  Pixel min/max: {real_batch.min():.3f} / {real_batch.max():.3f}")
print(f"  Pixel mean/std: {real_batch.mean():.3f} / {real_batch.std():.3f}")
print(f"  First 16 labels: {real_labels[:16].tolist()}")


### Step 2: Define generator and discriminator

The generator upsamples a 100-dimensional noise vector through four transposed convolutions with batch normalization and ReLU. The final layer uses `Tanh` and produces a 32x32 map; we center-crop to 28x28 to match MNIST.

The discriminator mirrors this with strided convolutions, LeakyReLU(0.2), and batch norm on all but the first layer. The output is one logit per image, paired with `BCEWithLogitsLoss` for numerical stability.


In [ ]:
class DCGANGenerator(nn.Module):
    def __init__(self, latent_dim: int = 100, ngf: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 8, ngf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            nn.ConvTranspose2d(ngf, 1, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = self.net(z.view(z.size(0), -1, 1, 1))
        return x[:, :, 2:30, 2:30]


class DCGANDiscriminator(nn.Module):
    def __init__(self, ndf: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 4, ndf * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf * 8),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(ndf * 8, 1, 4, 1, 0, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).view(-1)


### Step 3: Instantiate models and optimizers

We move both networks to `device`, count parameters, and sanity-check a forward pass on random noise. Adam with `lr=2e-4` and `betas=(0.5, 0.999)` matches the DCGAN paper.


In [ ]:
G = DCGANGenerator(LATENT_DIM).to(device)
D = DCGANDiscriminator().to(device)
criterion = nn.BCEWithLogitsLoss()
opt_g = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))

g_params = sum(p.numel() for p in G.parameters())
d_params = sum(p.numel() for p in D.parameters())
print("Architecture parameter counts")
print(f"  Generator:      {g_params:,}")
print(f"  Discriminator:  {d_params:,}")
print(f"  Total:          {g_params + d_params:,}")

with torch.no_grad():
    z_test = torch.randn(4, LATENT_DIM, device=device)
    fake_test = G(z_test)
    d_out = D(fake_test)
print("\nSanity check (batch of 4)")
print(f"  Fake image shape: {tuple(fake_test.shape)}")
print(f"  Discriminator logits: {d_out.cpu().tolist()}")


### Step 4: Train the DCGAN

Training alternates discriminator and generator updates. The discriminator learns to classify real images as 1 and fakes as 0. The generator is trained to fool the discriminator by pushing fake logits toward 1 (non-saturating objective).

We track per-epoch generator and discriminator loss, plus discriminator accuracy on real vs fake batches. Healthy training often shows D accuracy near 50-70% on both classes; if D reaches 100% on fakes, G is failing to learn.


In [ ]:
def batch_accuracy(logits: torch.Tensor, labels: torch.Tensor) -> float:
    preds = (torch.sigmoid(logits) > 0.5).float()
    return (preds == labels).float().mean().item()


g_losses, d_losses = [], []
d_real_accs, d_fake_accs = [], []
fixed_noise = torch.randn(64, LATENT_DIM, device=device)

for epoch in range(1, EPOCHS + 1):
    g_epoch, d_epoch = 0.0, 0.0
    real_acc_epoch, fake_acc_epoch = 0.0, 0.0
    for real, _ in loader:
        real = real.to(device)
        batch = real.size(0)
        real_labels = torch.ones(batch, device=device)
        fake_labels = torch.zeros(batch, device=device)

        D.zero_grad()
        d_real = D(real)
        loss_d_real = criterion(d_real, real_labels)
        z = torch.randn(batch, LATENT_DIM, device=device)
        fake = G(z).detach()
        d_fake = D(fake)
        loss_d_fake = criterion(d_fake, fake_labels)
        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        opt_d.step()

        G.zero_grad()
        z = torch.randn(batch, LATENT_DIM, device=device)
        fake = G(z)
        d_out = D(fake)
        loss_g = criterion(d_out, real_labels)
        loss_g.backward()
        opt_g.step()

        g_epoch += loss_g.item()
        d_epoch += loss_d.item()
        real_acc_epoch += batch_accuracy(d_real, real_labels)
        fake_acc_epoch += batch_accuracy(d_fake, fake_labels)

    g_losses.append(g_epoch / len(loader))
    d_losses.append(d_epoch / len(loader))
    d_real_accs.append(real_acc_epoch / len(loader))
    d_fake_accs.append(fake_acc_epoch / len(loader))
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"G loss {g_losses[-1]:.4f} | D loss {d_losses[-1]:.4f} | "
        f"D acc real {d_real_accs[-1]:.1%} | D acc fake {d_fake_accs[-1]:.1%}"
    )


### Step 5: Plot training curves

The loss plot summarizes the adversarial balance. Generator loss rising while discriminator loss falls can indicate mode collapse or a too-strong discriminator.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(range(1, EPOCHS + 1), g_losses, label="Generator", marker="o")
axes[0].plot(range(1, EPOCHS + 1), d_losses, label="Discriminator", marker="s")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("BCE loss")
axes[0].set_title("DCGAN training losses")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, EPOCHS + 1), d_real_accs, label="Real accuracy", marker="o")
axes[1].plot(range(1, EPOCHS + 1), d_fake_accs, label="Fake accuracy", marker="s")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Discriminator accuracy")
axes[1].set_title("Discriminator classification accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Step 6: Generate samples from fixed noise

Using the same noise vectors every epoch (stored in `fixed_noise`) makes it easy to see quality improve over training. We print pixel statistics for the generated batch and display an 8x8 grid.


In [ ]:
G.eval()
with torch.no_grad():
    fake_batch = G(fixed_noise).cpu()

print("Fixed-noise sample batch")
print(f"  Shape: {tuple(fake_batch.shape)}")
print(f"  Pixel min/max: {fake_batch.min():.3f} / {fake_batch.max():.3f}")
print(f"  Pixel mean/std: {fake_batch.mean():.3f} / {fake_batch.std():.3f}")
print("  Interpretation: mean near 0 and std below 1.0 are typical for Tanh outputs.")

grid = make_grid(fake_batch, nrow=8, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(6, 6))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.title("DCGAN samples (64 fixed noise vectors)")
plt.axis("off")
plt.tight_layout()
plt.show()
